# JSONL → JSON Batch Converter

Converts all `.jsonl` files in an **input folder** to `.json` files in an **output folder**.

- Original `.jsonl` files are **never deleted**
- Already-converted files are **skipped** on re-runs
- Progress is printed for every file

In [7]:
# ── Configuration ────────────────────────────────────────────────────────────
INPUT_FOLDER  = r"\Users\ecm3479\OneDrive - The University of Texas at Austin\Documents\Media Scrubbing Worksheets\JSONL"   # folder that contains your .jsonl files
OUTPUT_FOLDER = r"\Users\ecm3479\OneDrive - The University of Texas at Austin\Documents\Media Scrubbing Worksheets\JSON"    # folder where .json files will be saved
SKIP_EXISTING = True              # set False to re-convert already-done files
# ─────────────────────────────────────────────────────────────────────────────

In [8]:
import json
import os
from pathlib import Path

def convert_jsonl_to_json(jsonl_path: Path, json_path: Path) -> int:
    """
    Read a .jsonl file (one JSON object per line) and write a
    single .json file containing a JSON array of all records.

    Returns the number of records written.
    """
    records = []
    with jsonl_path.open("r", encoding="utf-8") as fh:
        for line_num, raw_line in enumerate(fh, start=1):
            line = raw_line.strip()
            if not line:          # skip blank lines
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                print(f"  ⚠  Skipping malformed line {line_num}: {exc}")

    with json_path.open("w", encoding="utf-8") as fh:
        json.dump(records, fh, ensure_ascii=False, indent=2)

    return len(records)


def batch_convert(input_folder: str, output_folder: str, skip_existing: bool = True):
    input_path  = Path(input_folder)
    output_path = Path(output_folder)

    if not input_path.exists():
        raise FileNotFoundError(f"Input folder not found: {input_path.resolve()}")

    output_path.mkdir(parents=True, exist_ok=True)

    jsonl_files = sorted(input_path.glob("*.jsonl"))

    if not jsonl_files:
        print("No .jsonl files found in", input_path.resolve())
        return

    print(f"Found {len(jsonl_files)} .jsonl file(s) in '{input_path.resolve()}'")
    print(f"Output folder : '{output_path.resolve()}'")
    print("-" * 60)

    converted = skipped = failed = 0

    for idx, jsonl_file in enumerate(jsonl_files, start=1):
        json_file = output_path / (jsonl_file.stem + ".json")

        prefix = f"[{idx:>{len(str(len(jsonl_files)))}}/{len(jsonl_files)}]"

        if skip_existing and json_file.exists():
            print(f"{prefix} SKIP     {jsonl_file.name}  (already converted)")
            skipped += 1
            continue

        try:
            n_records = convert_jsonl_to_json(jsonl_file, json_file)
            print(f"{prefix} OK       {jsonl_file.name}  →  {json_file.name}  ({n_records} records)")
            converted += 1
        except Exception as exc:
            print(f"{prefix} ERROR    {jsonl_file.name}  —  {exc}")
            failed += 1

    print("-" * 60)
    print(f"Done.  Converted: {converted}  |  Skipped: {skipped}  |  Failed: {failed}")
    print(f"Original .jsonl files are untouched in '{input_path.resolve()}'")

In [9]:
# ── Run the conversion ───────────────────────────────────────────────────────
batch_convert(INPUT_FOLDER, OUTPUT_FOLDER, skip_existing=SKIP_EXISTING)

Found 32 .jsonl file(s) in 'C:\Users\ecm3479\OneDrive - The University of Texas at Austin\Documents\Media Scrubbing Worksheets\JSONL'
Output folder : 'C:\Users\ecm3479\OneDrive - The University of Texas at Austin\Documents\Media Scrubbing Worksheets\JSON'
------------------------------------------------------------
[ 1/32] SKIP     r_appalachia_comments.jsonl  (already converted)
[ 2/32] SKIP     r_appalachia_posts.jsonl  (already converted)
[ 3/32] OK       r_asheville_comments.jsonl  →  r_asheville_comments.json  (827514 records)
[ 4/32] SKIP     r_asheville_posts.jsonl  (already converted)
[ 5/32] OK       r_austin_comments.jsonl  →  r_austin_comments.json  (1777359 records)
[ 6/32] OK       r_austin_posts.jsonl  →  r_austin_posts.json  (78591 records)
[ 7/32] SKIP     r_climate_comments.jsonl  (already converted)
[ 8/32] SKIP     r_climate_posts.jsonl  (already converted)
[ 9/32] SKIP     r_climatechange_comments.jsonl  (already converted)
[10/32] SKIP     r_climatechange_posts.jso